# Get Properties

Jupyter notbook interface for the automated collection of molecular descriptors and post-processing (i.e., Boltzmann average, min/max values, etc.) from Gaussian16 log files. Be sure to use the included environment before running.

__Contributors__
- Brittany C. Haas, PhD
- Melissa A. Hardy, PhD
- Jordan P. Liles, PhD
- James R. Howard, PhD

## Imports

In [ ]:
# stdlib
import os
import time
import shutil
import itertools

from pathlib import Path
from pprint import pprint
from multiprocessing import Pool

# Data wrangling
import numpy as np
import pandas as pd

# Jupyter
from IPython.display import SVG, display

# RDKit
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D

# Custom
import get_properties_functions_to_parallelize as gp
from utils import log_to_sdf, log_to_mol_file, _get_atom_map, mol_to_image

from goodvibes_functions import get_goodvibes_e
from new_property_functions import get_frontierorbs, get_polarizability, get_dipole, get_volume
from new_property_functions import get_SASA, get_nbo, get_nmr, get_distance, get_angles, get_dihedral
from new_property_functions import get_vbur_scan, get_sterimol_morfeus, get_chelpg, get_hirshfeld
from new_property_functions import get_pyramidalization
from utils import _read_in_mol_sdf_with_xyz_correction

# Initial settings
pd.options.display.max_columns = None
randomstate = 42

## Settings
In this section, point the Path `data` to the directory containing your G16 log files. The script will automatically convert them to .sdf files for further processing.

In [ ]:
# Point Path to the location of the log files
data = Path('./data/wolf_made_products')

# Define number of processors to use (set to number or os.cpu_count())
procs = os.cpu_count()

# Point path to where you want failed .log files to be moved
failed_dir = Path('./data/wolf_made_products_failed/')

# Turn on debug printing
debug = False

## Convert .log files to .mol files

In [ ]:
# Find the log files that do not have corresponding mol files
logs_to_convert = [x for x in data.glob('*.log') if not x.with_suffix('.mol').exists()]

print(f'[INFO] Converting {len(logs_to_convert)} logfiles to .mol files.')
if len(logs_to_convert) != 0:
    with Pool() as pool:
        map = pool.map(log_to_mol_file, logs_to_convert)

## Define Substructure for Parameterization
Provide and [SMARTS](https://www.daylight.com/dayhtml/doc/theory/theory.smarts.html) string or SMILES string that defines the substructure that you wish to parameterize. If you're unfamiliar with defining SMARTS patterns, begin by copying the SMILES directly from ChemDraw first. If that does not correspond to your desired substructure (e.g., hydrogens are missing), consult the [SMARTS documentation](https://www.daylight.com/dayhtml/doc/theory/theory.smarts.html).


In [ ]:
# Define a substructure common to all data files you are looking at
#substructure = Chem.MolFromSmarts('c:1:n:c:c:c:c:1')
substructure = Chem.MolFromSmarts('CS(C)=N')

im = mol_to_image(substructure, show_atom_indices=True)

display(SVG(data=im))

## Generate DataFrame containing files and atom numbers
Here we read in the `.sdf` files and identify the substructure pattern in each. These identified patterns will be assigned an atom number (1-indexed values) that we will eventually turn into a text label that is meaningful for parameterization.

In [ ]:
# Get the atom maps in parallel
with Pool(processes=procs) as p:
    results = p.starmap(_get_atom_map, zip(list(data.glob('*.mol')), itertools.repeat(substructure), itertools.repeat(debug)))

failed_files = []
new_results = []

for item in results:
    if item[1] is None:
        print(f'[WARNING] {item[0].name} is not readable with RDKit or had multiple substructure matches. Does the .mol exist?: {item[0].with_suffix(".mol").exists()}')
        failed_files.append(item[0])
    else:
        new_results.append(item)

# Place the atom numbers for the substructure for each log file into a dataframe
prelim_df = pd.DataFrame(new_results)
prelim_df.rename(columns={0: 'log_name'}, inplace=True)

#prelim_df = pd.DataFrame([Path(x).with_suffix('.log').name for x in failed_files])
#prelim_df.rename(columns={0: 'log_name'}, inplace=True)
#display(prelim_df)

# Save the prelim dataframe
prelim_df.to_csv('prelim_df.csv')

# Print the failed items
if len(failed_files) == 0:
    print(f'[INFO] All files were successfully read in.')
else:
    failed_dir.mkdir(exist_ok=True)
    print(f'[INFO] Files that were not able to be read with RDKit.')
    for f in failed_files:
        print(f)
        shutil.move(f, failed_dir / f.name)
        shutil.move(f.with_suffix('.log'), failed_dir / f.with_suffix('.log').name)

print(f'There are {len(failed_files)} failed files. There are {prelim_df.shape[0]} successfully read in files.')
display(prelim_df)
display()


## Rename columns to user-interpretable labels

This cell will randomly sample rows in your `prelim_df` and draw the molecules with several atoms replaced with `column headers`. These column headers should be reassigned to a user-defined label that is more meaningful for property collection.

In [ ]:
# Sample a small section of the molecules to view
sample = prelim_df.sample(n=3)

# Prepare in the molecules and labels
sampled_mols = []
sampled_labels = []
for i, row in sample.iterrows():
    # Read in the file as mol object
    file = Path(data / row['log_name']).with_suffix('.mol')
    file, mol = _read_in_mol_sdf_with_xyz_correction(file=file)

    if mol is None:
        print(f'[ERROR] Sampled a None mol!')
        continue

    for column_header in [x for x in row.keys() if 'log_name' not in str(x)]:
        for atom in mol.GetAtoms():
            # For each atom, set the property "atomNote" to a index+1 of the atom
            if atom.GetIdx() + 1 == row[column_header]:
                #atom.SetProp("atomNote", f'col: {column_header}#: {atom.GetIdx() + 1}')
                atom.SetProp("atomLabel", f'{column_header}')
    sampled_mols.append(mol)
    sampled_labels.append(row['log_name'])

# Make some drawing options
d2d = rdMolDraw2D.MolDraw2DSVG(2000, 2000)
dopts = d2d.drawOptions()
dopts.bondLineWidth = 1.4
image = Draw.MolsToGridImage(sampled_mols, molsPerRow=3, useSVG=False, legends=sampled_labels, drawOptions=dopts)
display(image)
display(sample)

# Print out a template dictionary
print({k: '' for k in prelim_df.columns if k != 'log_name'})

## Set an atom label dictionary

**NOTE: it is very important you assign these correctly otherwise the properties you collect will be for the wrong atoms and not produce meaningful correlations.** 

We recommend checking the numbering/headers for at least two different compounds. 

Numbering for different conformers of the same compounds will likely be the same (but may not be for symmetrical groups). These should be processed in later steps to average descriptors or compute deltas.

In [ ]:
# Create an atom label dictionary
atom_labels = {1: 'OXCO',
               2: 'OXC',
               3: 'OXN',
               4: 'OXC1',
               5: 'OXC2',
               6: 'OXC3',
               7: 'OXC4',
               8: 'OXC5',
               9: 'OXC6',
               10: 'OXC7',
               11: 'F',
               12: 'CMC1',
               13: 'CMC2',
               14: 'CMN',
               15: 'CMNO1', #
               16: 'CMNO2', #
               17: 'CMCOC',
               18: 'CMCOO',
               19: 'CMO',
               20: 'CMC3',
               21: 'CMC4',
               22: 'CMC5',
               23: 'CMC6',
               24: 'CMC7',
               25: 'CMC8'}

print(list(atom_labels.values()))

# Rename columns using the user input above
atom_map_df = prelim_df.rename(columns=atom_labels)
display(atom_map_df)

# Remove specific atoms if you wish to not collect descriptors for particular atoms
atom_map_df.drop(columns=[], inplace=True)

# df is what properties will be appended to, this creates a copy so that you have the original preserved
df = atom_map_df.copy(deep=True)

all_mapped_atoms = list(atom_labels.values())

### Save atom map to Excel (if desired)

In [ ]:
atom_map_df.to_csv('./results/atom_map.csv', index=False)

# Define Properties to Collect

Results with 1 processor: 6.01<br>
Results with 4 processor: 1.87<br>
Results with 8 processor: 1.28<br>
Results with 99 processor: 5.6 # Greater time if we exceed CPU count<br>

get_goodvibes_e<br>
1 proc = 156.751 seconds<br>
12 procs = 17.6 seconds<br>

Time comparisons for all properties:<br>
12 procs = 207.252 seconds<br>
6 procs = 253.864 seconds<br>
1 proc (old method) = 1000 seconds<br>

## Available property functions and how to call them: 

In [ ]:
t1 = time.time()
#df = atom_map_df.copy(deep=True)

#---------------GoodVibes Engergies---------------
#uses the GoodVibes 2021 Branch (Jupyter Notebook Compatible)
#calculates the quasi harmonic corrected G(T) and single point corrected G(T) and other thermodynamic properties
df = get_goodvibes_e(df, data_dir=data, temp=298.15, procs=procs)

#---------------Frontier Orbitals-----------------
#E(HOMO), E(LUMO), mu(chemical potential or negative of molecular electronegativity), eta(hardness/softness), omega(electrophilicity index)
df = get_frontierorbs(df, data_dir=data, procs=procs)

#---------------Polarizability--------------------
#Exact polarizability
df = get_polarizability(df, data_dir=data, procs=procs)

#---------------Dipole----------------------------
#Total dipole moment magnitude in Debye
df = get_dipole(df, data_dir=data, procs=procs)

#---------------Volume----------------------------
#Molar volume
#requires the Gaussian keyword = "volume" in the .com file
df = get_volume(df, data_dir=data, procs=procs)

#---------------SASA------------------------------
#Uses morfeus to calculate sovlent accessible surface area and the volume under the SASA
df = get_SASA(df, data_dir=data, procs=procs)

#---------------NBO-------------------------------
#natural charge from NBO
#requires the Gaussian keyword = "pop=nbo7" in the .com file
nbo_list = all_mapped_atoms           # Collecting NBOs of all mapped atoms
df = get_nbo(df, atom_list=nbo_list, data_dir=data, procs=procs)

#---------------NMR-------------------------------
#isotropic NMR shift
#requires the Gaussian keyword = "nmr=giao" in the .com file
nmr_list = all_mapped_atoms            # Collecting NMR shifts of all mapped atoms
df = get_nmr(df, atom_list=nmr_list, data_dir=data, procs=procs)

#---------------Distance--------------------------
#distance between 2 atoms
dist_list_of_lists = [['CMC1', 'OXC7']]
df = get_distance(df, dist_list=dist_list_of_lists, data_dir=data, procs=procs)

#---------------Vbur Scan-------------------------
#uses morfeus to calculate the buried volume at a series of radii (including hydrogens)
#inputs: dataframe, list of atoms, start_radius, end_radius, and step_size
#if you only want a single radius, put the same value for start_radius and end_radius (keep step_size > 0)
vbur_list = ['CMC1', 'OXC7']             # Collecting Vbur shifts of all mapped atoms
df = get_vbur_scan(df,
                   data_dir=data,
                   a_list=vbur_list,
                   start_r=2,
                   end_r=4,
                   step_size=0.5,
                   procs=procs)

#---------------Sterimol morfeus------------------
#uses morfeus to calculate Sterimol L, B1, and B5 values
#NOTE: this is much faster than the corresponding DBSTEP function (recommendation: use as default/if you don't need Sterimol2Vec)
sterimol_list_of_lists = [['CMC1', 'OXC7'], ['OXC7', 'CMC1']]
df = get_sterimol_morfeus(df, data_dir=data, sterimol_list=sterimol_list_of_lists, radius=None, procs=procs)

#---------------Buried Sterimol-------------------
#uses morfeus to calculate Sterimol L, B1, and B5 values within a given sphere of radius r_buried
#atoms outside the sphere + 0.5 vdW radius are deleted and the Sterimol vectors are calculated
#for more information: https://kjelljorner.github.io/morfeus/sterimol.html
sterimol_list_of_lists = [['CMC1', 'OXC7'], ['OXC7', 'CMC1']]
df = get_sterimol_morfeus(df, data_dir=data, sterimol_list=sterimol_list_of_lists, radius=5.5, procs=procs)

#---------------ChelpG----------------------------
#ChelpG ESP charge
#requires the Gaussian keyword = "pop=chelpg" in the .com file
a_list = all_mapped_atoms
df = get_chelpg(df, atom_list=a_list, data_dir=data, procs=procs)

#---------------Hirshfeld-------------------------
#Hirshfeld charge, CM5 charge, Hirshfeld atom dipole
#requires the Gaussian keyword = "pop=hirshfeld" in the .com file
a_list = all_mapped_atoms
df = get_hirshfeld(df, atom_list=a_list, data_dir=data, procs=procs)

display(df)
df.to_csv('./results/wolf_combined_product_individual_properties.csv', index=False)

print(f'Done. Time:\t{round(time.time() - t1, 3)} seconds')

## Parallelized functions

In [ ]:
#---------------Dihedral--------------------------
#dihedral angle between 4 atoms
dihedral_list_of_lists = [['C1','N', 'C1_prime','C2_prime']]
df = get_dihedral(df, dihedral_list=dihedral_list_of_lists, data_dir=data, procs=procs)

#---------------Angle-----------------------------
#angle between 3 atoms
angle_list_of_lists = [['S1', 'P', 'O'], ['S2', 'P', 'O'], ['S3', 'P', 'O']]
df = get_angles(df, angle_list=angle_list_of_lists, data_dir=data, procs=procs)

#---------------NBO-------------------------------
#natural charge from NBO
#requires the Gaussian keyword = "pop=nbo7" in the .com file
nbo_list = ['C14', 'C1', 'C', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'N', 'C7', 'C2', 'C3', 'C4', 'C5', 'C6']            # Collecting NBOs of all mapped atoms
df = get_nbo(df, atom_list=nbo_list, data_dir=data, procs=procs)

#---------------NMR-------------------------------
#isotropic NMR shift
#requires the Gaussian keyword = "nmr=giao" in the .com file
nmr_list = ['C14', 'C1', 'C', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'N', 'C7', 'C2', 'C3', 'C4', 'C5', 'C6']           # Collecting NMR shifts of all mapped atoms
df = get_nmr(df, atom_list=nmr_list, data_dir=data, procs=procs)

#---------------Distance--------------------------
#distance between 2 atoms
#dist_list_of_lists = [['N', 'C1'], ['N', 'C1_prime'], ['C1', 'C2'], ['C1_prime', 'C2_prime'], ['C2', 'C3'], ['C2_prime', 'C3']]
#df = get_distance(df, dist_list=dist_list_of_lists, data_dir=data, procs=procs)

#---------------Angle-----------------------------
#angle between 3 atoms
#angle_list_of_lists = [['C1', 'N', 'C1_prime']]
#df = get_angles(df, angle_list=angle_list_of_lists, data_dir=data, procs=procs)

#---------------Vbur Scan-------------------------
#uses morfeus to calculate the buried volume at a series of radii (including hydrogens)
#inputs: dataframe, list of atoms, start_radius, end_radius, and step_size
#if you only want a single radius, put the same value for start_radius and end_radius (keep step_size > 0)
vbur_list = ['C14', 'C1', 'C', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'N', 'C7', 'C2', 'C3', 'C4', 'C5', 'C6']             # Collecting Vbur shifts of all mapped atoms
df = get_vbur_scan(df,
                   data_dir=data,
                   a_list=vbur_list,
                   start_r=2,
                   end_r=4,
                   step_size=0.5,
                   procs=procs)

#---------------Sterimol morfeus------------------
#uses morfeus to calculate Sterimol L, B1, and B5 values
#NOTE: this is much faster than the corresponding DBSTEP function (recommendation: use as default/if you don't need Sterimol2Vec)
sterimol_list_of_lists = [["C11", "C8"]]
df = get_sterimol_morfeus(df, data_dir=data, sterimol_list=sterimol_list_of_lists, radius=None, procs=procs)

#---------------Buried Sterimol-------------------
#uses morfeus to calculate Sterimol L, B1, and B5 values within a given sphere of radius r_buried
#atoms outside the sphere + 0.5 vdW radius are deleted and the Sterimol vectors are calculated
#for more information: https://kjelljorner.github.io/morfeus/sterimol.html
sterimol_list_of_lists = [["C11", "C8"]]
df = get_sterimol_morfeus(df, data_dir=data, sterimol_list=sterimol_list_of_lists, radius=5.5, procs=procs)

#---------------ChelpG----------------------------
#ChelpG ESP charge
#requires the Gaussian keyword = "pop=chelpg" in the .com file
a_list = ['C14', 'C1', 'C', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'N', 'C7', 'C2', 'C3', 'C4', 'C5', 'C6']
df = get_chelpg(df, atom_list=a_list, data_dir=data, procs=procs)

#---------------Hirshfeld-------------------------
#Hirshfeld charge, CM5 charge, Hirshfeld atom dipole
#requires the Gaussian keyword = "pop=hirshfeld" in the .com file
a_list = ['C14', 'C1', 'C', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'N', 'C7', 'C2', 'C3', 'C4', 'C5', 'C6']
df = get_hirshfeld(df, atom_list=a_list, data_dir=data, procs=procs)

#---------------Pyramidalization------------------
#uses morfeus to calculate pyramidalization based on the 3 atoms in closest proximity to the defined atom
#collects values based on two definitions of pyramidalization
#details on these values can be found here: https://kjelljorner.github.io/morfeus/pyramidalization.html
pyr_list = ['P']             # Collecting pyramidalization of all mapped atoms
df = get_pyramidalization(df, atom_list=pyr_list, data_dir=data, procs=procs, )


## Non-parallelized Functions
These properties are less commonly extracted and are not currently parallelized.

In [ ]:
df = atom_map_df
#---------------Time----------------------------------
#returns the total CPU time and total Wall time (not per subjob) because we are pioneers
#if used in summary df, will give the average (not Boltzmann average) in the Boltzmann average column
#df = gp.get_time(df)

#---------------IR--------------------------------
#CAUTION: CANNOT ACCURATELY IDENTIFY ATOM STRETCHES IN MOST CASES (strugges if there is more than one stretch in the defined range)
#IR frequencies and intensities in a specific range (for specific atoms)
#requires the Gaussian keyword = "freq=noraman" in the .com file
#inputs: dataframe, atom1, atom2, frequency_min, frequency_max, intensity_min, intensity_max, threshold
#if you want to collect multiple IR frequencies, you will need to copy/paste this function for each stretch
#we recommend a threshold of 0.0 (may have to adjust)
#df = gp.get_IR(df, "C1", "O2", 1700, 1900, 100, 800, 0.0)

#---------------Sterimol DBSTEP-------------------
# Uses DBSTEP to calculate Sterimol L, B1, and B5 values
# Default grid point spacing (0.05 Angstrom) is used (can use custom spacing or vdw radii in the get_properties_functions script)
#more info here: https://github.com/patonlab/DBSTEP
#NOTE: this takes longer than the morfeus function (recommendation: only use this if you need Sterimol2Vec)
#sterimol_list_of_lists = [["O3", "H5"]]
#df = gp.get_sterimol_dbstep(df, sterimol_list_of_lists)

#---------------Sterimol2Vec----------------------
# Sses DBSTEP to calculate Sterimol Bmin and Bmax values at intervals from 0 to end_radius, with a given step_size
# Default grid point spacing (0.05 Angstrom) is used (can use custom spacing or vdw radii in the get_properties_functions script)
#more info here: https://github.com/patonlab/DBSTEP
#inputs: dataframe, list of atom pairs, end_radius, and step_size
#sterimol2vec_list_of_lists = [["C1", "C4"]]
#df = gp.get_sterimol2vec(df, sterimol2vec_list_of_lists, 1, 1.0)

#---------------Plane Angle-----------------------
#plane angle between 2 planes (each defined by 3 atoms)
#planeangle_list_of_lists = [["C3", "C2", "N"]]
#df = gp.get_planeangle(df, planeangle_list_of_lists)

#---------------Pyramidalization------------------
#uses morfeus to calculate pyramidalization based on the 3 atoms in closest proximity to the defined atom
#collects values based on two definitions of pyramidalization
#details on these values can be found here: https://kjelljorner.github.io/morfeus/pyramidalization.html
pyr_list = ['C1','N', 'C1_prime','C2_prime', 'C3', 'C2']             # Collecting pyramidalization of all mapped atoms
df = gp.get_pyramidalization(df, pyr_list)


## Save collected properties to Excel

Helpful to save here in case the Notebook crashes or if you want to add more properties before post-processsing. Can be read in at 5.1.1.

In [ ]:
writer = pd.ExcelWriter('wolf_combined_product_individual_properties.xlsx')
df.to_excel(writer)
writer.save()

# Post-processing

## Generating a list of compounds that have conformational ensembles

In [ ]:
# Read in the conformer specific properties
df = pd.read_excel('./sdi-props-processed.xlsx', header=0, sheet_name='Sheet1')
# Define the separator that divides the file names
# and the conformer number
separator = "_"

# Drop anything that's not a property we want to average
columns_to_drop = all_mapped_atoms
#columns_to_drop = []

# Energy column header for boltzmann weighting and lowest energy conformer
energy_col_header = "G(T)_spc(Hartree)"

# specify energy cutoff in kcal/mol to remove conformers above this value before post-processing
energy_cutoff = 5.0

# set to true if you'd like to see info on the nunmber of conformers removed for each compound
verbose = True

def split_compound_name(s: str, delimiter: str, return_key: int = 0) -> str:
    '''
    '''
    s = Path(s).stem
    return str(s.split(delimiter)[return_key])

df['PREFIX'] = df['file_name'].apply(split_compound_name, delimiter='_', return_key=0)
df['SUFFIX'] = df['file_name'].apply(split_compound_name, delimiter='_', return_key=1)
df.sort_values('PREFIX', inplace=True)

display(df)

# Make a dictionary of compound id values and number of conformers
compound_id_conf_number = {}
for compound_name in set(df['PREFIX'].to_list()):
    compound_id_conf_number[compound_name] = len(df[df['PREFIX'] == compound_name])

## Removing entries for which there is no thermochemical data

It appears that some log files were not able to be opened! In this section, we will remove the entries for which there is no energy_col_header value. In the current version of the script, this value is set to 'no data'

In [ ]:
# Drop any row with 'no data'
to_remove = []
to_keep = []
for i, row in df.iterrows():
    if any([value == 'no data' for value in row]):
        to_remove.append(row)
    else:
        to_keep.append(row)

# Display the ones that failed
print(f'[INFO] The following DataFrame contains log files for which some properties were "no data"')
display(pd.DataFrame(to_remove))

# Make a new dataframe of the successfully calculated values
df = pd.DataFrame(to_keep)

# Convert everything but log_name to float
df[[x for x in df.columns if x not in ['log_name', 'PREFIX', 'SUFFIX', 'file_name']]] = df[[x for x in df.columns if x not in ['log_name', 'PREFIX', 'SUFFIX', 'file_name']]].astype(float)

print(f'[INFO] The following DataFrame contains log files for which all properties were computed.')
display(df)

## Post-processing to get properties for each compound

In [ ]:
# Set the Boltzmann-weighting temperature
temperature = 298.15

# Define columns that are not processed for summary properties
ignore_columns = ['log_name',
                  'PREFIX',
                  'SUFFIX',
                  'G(Hartree)',
                  '∆G(Hartree)',
                  '∆G(kcal/mol)',
                  'e^(-∆G/RT)',
                  'Mole Fraction',
                  'CPU_time_total(hours)',
                  'Wall_time_total(hours)',
                  'E_spc (Hartree)',
                  'H_spc(Hartree)',
                  'T',
                  'T*S',
                  'T*qh_S',
                  'ZPE(Hartree)',
                  'qh_G(T)_spc(Hartree)',
                  'Neutral SPE (Eh)', # Added these for redox computations for the pyridine project
                  'Neutral G-El (Eh)',
                  'Neutral dGsolv (Eh)',
                  'Oxidized SPE (Eh)',
                  'Oxidized G-El (Eh)',
                  'Oxidized dGsolv (Eh)',
                  'Solvent-corrected-G (Ox)',
                  'Solvent-corrected-G (Neut)',
                  'dG reaction (Eh)',
                  'dG reaction (kcal/mol)',
                  'file_name'
                  ]

# Place this in try/except if only global properties are collected
try:
    ignore_columns.extend(list(atom_labels.values()))
except NameError as e:
    print(f'[WARNING] No atom labels were specified to ignore/drop.')

try:
    df.drop(columns=columns_to_drop, inplace=True)
except KeyError as e:
    print(f'[WARNING] Could not drop {columns_to_drop} from DataFrame.')

# List of dataframes to concat after everything is done
results = []

# Iterate over all compounds
for compound_identifier, number_of_confs in compound_id_conf_number.items():

    #if verbose:
    #    print(f'[VERBOSE] Working on {compound_identifier} (n_confs={number_of_confs})')

    # Read in the PREFIX (which unifies the conformers of a single molecule) as its respective type
    if df['PREFIX'].dtype == str:
        valuesdf = df[df['PREFIX'].astype(str) == str(compound_identifier)].copy(deep=True)
    elif df['PREFIX'].dtype == float:
        valuesdf = df[df['PREFIX'].astype(float) == float(compound_identifier)].copy(deep=True)
    elif df['PREFIX'].dtype == int:
        valuesdf = df[df['PREFIX'].astype(int) == int(compound_identifier)].copy(deep=True)
    elif df['PREFIX'].dtype == object:
        valuesdf = df[df['PREFIX'].astype(str) == str(compound_identifier)].copy(deep=True)
    else:
        raise ValueError(f'Dtype of identification columns (PREFIX and SUFFIX) is not understood: {df["PREFIX"].dtype}')

    if len(valuesdf) == 0:
        print(f'[ERROR] No conformer data could be found for {compound_identifier}')
        continue
    elif len(valuesdf) != number_of_confs:
        print(f'[ERROR] Mismatch in number of confs for {compound_identifier}. Should be {number_of_confs} but found {len(valuesdf)}')
        continue
        assert number_of_confs == len(valuesdf)

    # Check for 'no data' string
    no_data_check = valuesdf.copy(deep=True)
    valuesdf.replace('no data', np.NAN, inplace=True)
    valuesdf.dropna(axis=0, how='any')

    if valuesdf.shape[0] != no_data_check.shape[0]:
        print(f'[WARNING] Some rows (conformers) in your data frame had no data')

    # Get the columns that we want to min/max/average
    columns_to_process = [x for x in valuesdf.columns if x not in columns_to_drop and x != 'log_name']

    # Calculate the summary properties based on all conformers (Boltzmann Average, Minimum, Maximum, Boltzmann Weighted Std)
    # Get the relative energy in Hartrees and kcal
    valuesdf["∆G(Hartree)"] = valuesdf[energy_col_header].astype(float) - valuesdf[energy_col_header].astype(float).min()
    valuesdf["∆G(kcal/mol)"] = valuesdf["∆G(Hartree)"] * 627.509

    # Compute the Boltzmann weights
    valuesdf["e^(-∆G/RT)"] = (valuesdf["∆G(kcal/mol)"] * -1000) / (1.987204 * temperature) #R is in cal/(K*mol)
    valuesdf['e^(-∆G/RT)'] = valuesdf['e^(-∆G/RT)'].astype(float).apply(np.exp)
    valuesdf["Mole Fraction"] = valuesdf["e^(-∆G/RT)"] / valuesdf["e^(-∆G/RT)"].sum()

    # Apply the energy cutoff
    valuesdf.drop(valuesdf[valuesdf["∆G(kcal/mol)"] >= energy_cutoff].index, inplace=True)
    valuesdf = valuesdf.reset_index(drop = True)

    #if verbose:
    #    print(f'[VERBOSE] Initial # conformers: {number_of_confs} After energy cutoff: {len(valuesdf)}')

    summary_properties = {}
    summary_properties['log_name'] = compound_identifier

    for column in valuesdf:

        # Skip the columns to be ignored
        if column in ignore_columns:
            continue

        #if verbose:
        #    print(f'[VERBOSE] Working on property {column}')

        # Get min max range
        summary_properties[f'{column}_min'] = valuesdf[column].astype(float).min()
        summary_properties[f'{column}_max'] = valuesdf[column].astype(float).max()
        summary_properties[f'{column}_range'] = valuesdf[column].astype(float).max() - valuesdf[column].astype(float).min()

        # Get boltz
        summary_properties[f'{column}_Boltz'] = (valuesdf[column].astype(float) * valuesdf["Mole Fraction"].astype(float)).sum()

        # Get weighted stdev
        # https://www.statology.org/weighted-standard-deviation-excel/
        #if len(valuesdf) == 1:
        #    summary_properties[f'{column}_Boltz_stdev'] = 0
        #else:
        #    summary_properties[f'{column}_Boltz_stdev'] = np.std(valuesdf[column].astype(float).to_list())

        if len(valuesdf) == 1:
            summary_properties[f'{column}_Boltz_stdev'] = 0
        else:
            boltz = (valuesdf[column] * valuesdf["Mole Fraction"]).sum()
            delta_values_sq = []

            # Get a list of the weights
            w = valuesdf["Mole Fraction"].astype(float).to_numpy()
            delta_values_sq = valuesdf[column].astype(float).to_numpy()
            delta_values_sq = np.array([(x - boltz)**2 for x in delta_values_sq])

            wstdev = np.sqrt( (np.average(delta_values_sq, weights=w)) / (((len(w) - 1) / len(w)) * np.sum(w)) )
            summary_properties[f'{column}_Boltz_stdev'] = wstdev

        # Get lowest energy conformer (LEC)
        lec_df = valuesdf[valuesdf[energy_col_header].astype(float) == valuesdf[energy_col_header].astype(float).min()]
        if len(lec_df) != 1:

            # Check if all of the values are the same for the property we are looking at
            if not all([x == lec_df[column].astype(float).to_list()[0] for x in lec_df[column].astype(float).to_list()]):
                _lowest_energy_debug = valuesdf[energy_col_header].astype(float).min()
                print(f'[WARNING] Multiple LECs of the same energy ({round(_lowest_energy_debug, 5)} {energy_col_header}) were found, but they have different properties {lec_df[column].astype(float).to_list()}. Taking the average.')

            # Take the average either way since it's a low cost operation
            summary_properties[f'{column}_low_E'] = np.average(lec_df[column].astype(float).to_list())

        else:
            # Take the first and only value
            summary_properties[f'{column}_low_E'] = lec_df[column].astype(float).values[0]

    results.append(pd.Series(summary_properties))

summary_properties = pd.DataFrame(results)
display(summary_properties)


#------------------------------EDIT THIS SECTION IF YOU WANT A SPECIFIC CONFORMER----------------------------------
    #if you want all properties for a conformer with a particular property (i.e. all properties for the Vbur_min conformer)
    #this template can be adjusted for min/max/etc.

    #find the index for the min or max column:
    #ensemble_min_index = valuesdf[valuesdf["log_name"] == "Ensemble Minimum"].index.tolist()

    #find the min or max value of the property (based on index above)
    #saves the value in a list (min_value) with one entry (this is why we call min_value[0])
    #min_value = valuesdf.loc[ensemble_min_index, "%Vbur_C4_3.0Å"].tolist()
    #vbur_min_index = valuesdf[valuesdf["%Vbur_C4_3.0Å"] == min_value[0]].index.tolist()

    #copy the row to a new_row with the name of the log changed to Property_min_conformer
    #new_row = valuesdf.loc[vbur_min_index[0]]
    #new_row['log_name'] = "%Vbur_C4_3.0Å_min_Conformer"
    #valuesdf =  valuesdf.append(new_row, ignore_index=True)
#--------------------------------------------------------------------------------------------------------------------



## Saving the Results

In [ ]:
summary_properties.to_csv(f'./test_hannah_differences_{temperature}_old_R_no_averaging_lecs_max.csv', index=False)

## Making Derivative Properties
Because certain molecules have symmetry elements that make assignment of descriptors to atoms challenging, it can be advantageous to "summarize" the summary properties. For instance, the two alpha carbons of a ketone might be important for a particular substitution reaction, but it is can be difficult to assign descriptors to a particular alpha carbon across a series of ketones. Thus, we can perform mathematical operations on the descriptors for a better description of the molecule's reactivity. The alpha carbons can be further processed to describe the __average__, __min__, or __max__ value of a property across both alpha carbons. Further derivitization like __delta__ could be envisioned where the difference in a property across descriptor-equivalent atoms is important. 
<br>
<br>
This section uses the definitions specified by the user to add descriptors based on these principles.

In [ ]:
# Read in the summary properties
df = pd.read_csv('./p_oxide_properties.csv')
display(df)

ignore_columns = ['log_name',
                  'PREFIX',
                  'SUFFIX',
                  'G(Hartree)',
                  '∆G(Hartree)',
                  '∆G(kcal/mol)',
                  'e^(-∆G/RT)',
                  'Mole Fraction',
                  'CPU_time_total(hours)',
                  'Wall_time_total(hours)',
                  'E_spc (Hartree)',
                  'H_spc(Hartree)',
                  'T',
                  'T*S',
                  'T*qh_S',
                  'ZPE(Hartree)',
                  'qh_G(T)_spc(Hartree)',
                  ]

# Get a list of descriptors that are candidates for derivative properties
descriptors = [x for x in df.columns if x not in ignore_columns]

# Summary property labels that will be considered
suffixes = ['min', 'max', 'lec', 'boltz']

descriptors_to_symmetrize = [
    # Single atom properties (NBO, ChelpG, Hirsh CM5, Hirsh dipole, NMR)
    ['NBO_charge_S1', 'NBO_charge_S2', 'NBO_charge_S3'],
    ['ChelpG_charge_S1', 'ChelpG_charge_S2', 'ChelpG_charge_S3'],
    ['Hirsh_CM5_charge_S1', 'Hirsh_CM5_charge_S2', 'Hirsh_CM5_charge_S3'],
    ['Hirsh_atom_dipole_S1', 'Hirsh_atom_dipole_S2', 'Hirsh_atom_dipole_S3'],
    ['NMR_shift_S1', 'NMR_shift_S2', 'NMR_shift_S3'],

    # Distances
    ['distance_S1_P(Å)', 'distance_S2_P(Å)', 'distance_S3_P(Å)'],

]

to_drop = []

# For the two chemically equivalent nitrogens
# Find the minimum energy conf properties for N and N'
# Then select the min, max of those two values

#TODO Dont nest loops
for descriptor_group in descriptors_to_symmetrize:

    for s in suffixes:

        # Get all the descriptors with the right suffix
        to_sym = [x for x in descriptors if s in x and any([z in x for z in descriptor_group])]

        if to_sym == []:
            continue

        new_descriptor_base = [x.replace('S1', 'sub') for x in to_sym]
        new_descriptor_base = [x.replace('S2', 'sub') for x in new_descriptor_base]
        new_descriptor_base = [x.replace('S3', 'sub') for x in new_descriptor_base]
        new_descriptor_base = list(set(new_descriptor_base))

        if len(new_descriptor_base) != 1:
            raise ValueError(new_descriptor_base, len(new_descriptor_base))

        new_descriptor_base = new_descriptor_base[0]

        print(f'Going to symmetrize {to_sym}. Base: {new_descriptor_base}')

        df[f'{new_descriptor_base}_min'] = df[to_sym].min(axis=1)
        df[f'{new_descriptor_base}_max'] = df[to_sym].max(axis=1)
        df[f'{new_descriptor_base}_average'] = df[to_sym].mean(axis=1)
        if len(to_sym) == 2:
            df[f'{new_descriptor_base}_delta'] = abs(df[to_sym[0]] - df[to_sym[1]])
        else:
            df[f'{new_descriptor_base}_range'] = abs(df[to_sym].max(axis=1) - df[to_sym].min(axis=1))

        to_drop.extend(to_sym)

df.to_csv('./p_oxide_summary_properties.csv')

